# MCP Gateway Agent Demo

Same agent, same code. Only the **URL** changes.

**The only change between "Managed" and "Clone" is the MCP endpoint URL.**

| Scenario | URL | Result |
|----------|-----|--------|
| Managed MCP (1) | `WORKSPACE_HOST/api/2.0/mcp/genie/{space_id}` | Direct — single question |
| Managed MCP (7) | `WORKSPACE_HOST/api/2.0/mcp/genie/{space_id}` | Direct — 7 in parallel → errors/rate limit |
| Clone MCP (7, 1st) | `APP_HOST/api/2.0/mcp/genie/{gateway_id}` | Via Gateway — queue + retry |
| Clone MCP (7, 2nd) | `APP_HOST/api/2.0/mcp/genie/{gateway_id}` | Instant — semantic cache |

In [ ]:
%pip install 'openai-agents[mcp]' -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("app_url", "", "App URL")
dbutils.widgets.text("space_id", "", "Genie Space ID")
dbutils.widgets.text("gateway_id", "", "Gateway ID")
dbutils.widgets.text("model", "databricks-claude-sonnet-4", "Model Serving Endpoint")

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
WORKSPACE_HOST = w.config.host.rstrip("/")
TOKEN          = dbutils.secrets.get(scope="genie-cache", key="oauth_token")
APP_HOST       = dbutils.widgets.get("app_url").rstrip("/")
GENIE_SPACE_ID = dbutils.widgets.get("space_id")
GATEWAY_ID     = dbutils.widgets.get("gateway_id")
MODEL          = dbutils.widgets.get("model")

MANAGED_URL = f"{WORKSPACE_HOST}/api/2.0/mcp/genie/{GENIE_SPACE_ID}"
CLONE_URL   = f"{APP_HOST}/api/2.0/mcp/genie/{GATEWAY_ID}"

print(f"Managed: {MANAGED_URL}")
print(f"Clone:   {CLONE_URL}")

In [ ]:
import asyncio, time, logging
from openai import AsyncOpenAI
from agents import Agent, Runner
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from agents.mcp import MCPServerStreamableHttp

logging.getLogger("openai.agents").setLevel(logging.ERROR)


class _StripStrictClient(AsyncOpenAI):
    """Wrapper that removes 'strict' from tool definitions before sending.
    Databricks model serving does not support this OpenAI-specific field."""

    class _Completions:
        def __init__(self, inner):
            self._inner = inner

        async def create(self, **kwargs):
            for tool in kwargs.get("tools", []) or []:
                if isinstance(tool, dict) and "function" in tool:
                    tool["function"].pop("strict", None)
            return await self._inner.create(**kwargs)

    class _Chat:
        def __init__(self, completions):
            self.completions = completions

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.chat = self._Chat(self._Completions(super().chat.completions))


oai = _StripStrictClient(base_url=f"{WORKSPACE_HOST}/serving-endpoints", api_key=TOKEN)
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=oai)

AUTH = {"Authorization": f"Bearer {TOKEN}"}

AGENT_INSTRUCTIONS = (
    "Answer the user's question using the genie space tools. "
    "Be concise: reply with the answer in 1-2 sentences. "
    "Do not repeat raw data tables in your response."
)

questions = [
    "What are the top 3 nations by total revenue?",
    "How many orders were placed in January 1994?",
    "What is the average account balance of customers in the BUILDING segment?",
    "What is the total number of suppliers?",
    "Total number of lineitems with quantity greater than 40",
    "Revenue by year for the ASIA region",
    "How many distinct part types exist?",
]

async def ask(url, question, label=""):
    t0 = time.time()
    try:
        async with MCPServerStreamableHttp(params={"url": url, "headers": AUTH}, client_session_timeout_seconds=120) as mcp:
            agent = Agent(
                name="analyst",
                instructions=AGENT_INSTRUCTIONS,
                model=model,
                mcp_servers=[mcp],
                mcp_config={"convert_schemas_to_strict": False},
            )
            result = await Runner.run(agent, question, max_turns=30)
            elapsed = round(time.time() - t0, 1)
            output = result.final_output[:200]
            print(f"  {label}[{elapsed}s] OK | {output}")
            return {"status": "OK", "output": output, "time": elapsed}
    except Exception as e:
        elapsed = round(time.time() - t0, 1)
        error = str(e)[:150]
        tag = "429" if "429" in error or "rate" in error.lower() else "ERROR"
        print(f"  {label}[{elapsed}s] {tag} | {error}")
        return {"status": tag, "output": error, "time": elapsed}


async def ask_batch(url, qs, stagger=0.5):
    """Run questions with a small stagger to avoid overwhelming the server."""
    async def _delayed(i, q):
        await asyncio.sleep(i * stagger)
        return await ask(url, q, label=f"Q{i+1} ")
    return await asyncio.gather(*[_delayed(i, q) for i, q in enumerate(qs)])

print("Ready.")

## Scenario A: Managed MCP — single question (baseline)

Direct to the workspace Genie MCP endpoint — no caching, no queue. One question to establish the baseline.

In [ ]:
print(f"URL: {MANAGED_URL}\n")
result_a = await ask(MANAGED_URL, questions[0], label="Q1 ")

## Scenario B: Managed MCP — 7 questions in parallel (rate limit)

Same managed endpoint, 7 concurrent agents. Genie caps at 5 QPM → expect failures.

In [ ]:
print(f"URL: {MANAGED_URL}\n")
t0 = time.time()
results_b = await ask_batch(MANAGED_URL, questions, stagger=0.3)
time_b = round(time.time() - t0, 1)

ok = sum(1 for r in results_b if r["status"] == "OK")
errors = sum(1 for r in results_b if r["status"] != "OK")
print(f"\n{ok} completed, {errors} failed | Total: {time_b}s")

## Scenario C: Clone MCP — 7 questions in parallel (gateway)

Same 7 questions, same agent. Only the URL points to the Gateway. Queue + retry handles the rate limit.

In [ ]:
print(f"URL: {CLONE_URL}\n")
t0 = time.time()
results_c = await ask_batch(CLONE_URL, questions, stagger=0.5)
time_c = round(time.time() - t0, 1)

ok = sum(1 for r in results_c if r["status"] == "OK")
print(f"\n{ok}/{len(questions)} completed | Total: {time_c}s")

## Scenario D: Clone MCP — 7 questions again (cache hits)

Same 7 questions, same URL. All should hit the **semantic cache** — instant responses.

In [ ]:
print(f"URL: {CLONE_URL}\n")
t0 = time.time()
results_d = await ask_batch(CLONE_URL, questions, stagger=0.3)
time_d = round(time.time() - t0, 1)

ok = sum(1 for r in results_d if r["status"] == "OK")
print(f"\n{ok}/{len(questions)} completed (cache hits) | Total: {time_d}s")

## Comparison

In [ ]:
def _tag(r):
    return r["status"]

print(f"{'Question':55s} | {'Managed':>8s} | {'Gateway':>8s} | {'Cached':>8s}")
print(f"{'-'*87}")
for i, q in enumerate(questions):
    print(f"{q:55s} | {_tag(results_b[i]):>8s} | {_tag(results_c[i]):>8s} | {_tag(results_d[i]):>8s}")

print()
b_ok = sum(1 for r in results_b if r["status"] == "OK")
c_ok = sum(1 for r in results_c if r["status"] == "OK")
d_ok = sum(1 for r in results_d if r["status"] == "OK")

print(f"B - Managed (7 parallel):  {b_ok}/{len(questions)} completed | {time_b}s")
print(f"C - Gateway (7 parallel):  {c_ok}/{len(questions)} completed | {time_c}s")
print(f"D - Gateway (cache hits):  {d_ok}/{len(questions)} completed | {time_d}s")
print()
print("Same agent. Only the URL changed.")